Напишем пользовательский класс ANFIS

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

class ANFIS(nn.Module):
    def __init__(self, n_inputs, n_rules):
        super(ANFIS, self).__init__()
        self.n_inputs = n_inputs
        self.n_rules = n_rules

        self.centers = nn.Parameter(torch.rand(n_inputs, n_rules))
        self.sigmas = nn.Parameter(torch.rand(n_inputs, n_rules))

        self.rule_weights = nn.Linear(n_rules, 1, bias=False)

    def gaussian_membership(self, x):
        return torch.exp(-((x.unsqueeze(-1) - self.centers) ** 2 / (2 * self.sigmas ** 2)))

    def forward(self, x):
        membership_values = self.gaussian_membership(x)  # (batch_size, n_inputs, n_rules)

        rule_activation = torch.prod(membership_values, dim=1)  # (batch_size, n_rules)

        rule_activation_normalized = rule_activation / rule_activation.sum(dim=1, keepdim=True)

        output = self.rule_weights(rule_activation_normalized)  # (batch_size, 1)
        return output.squeeze()


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np


X = np.random.randn(1000000)


lower_bound = -np.pi / 2.01
upper_bound = np.pi / 2.01


X = X[(X > lower_bound) & (X < upper_bound)]
y = np.tan(X)

Xtr, Xtst, ytr, ytst = train_test_split(X, y)

In [ ]:
class Dataset():
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)


    def __len__(self):
        return self.X.shape[0]


    def __getitem__(self, index):
        return self.X[index], self.y[index]


In [ ]:
train_dataset = Dataset(Xtr, ytr)
test_dataset = Dataset(Xtst, ytst)
from torch.utils.data import DataLoader
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False)

Создадим объект класса ANFIS с 10 правилами

In [ ]:
n_inputs = 1
n_rules = 10

model = ANFIS(n_inputs, n_rules)


В качестве критерия оптимальности используем MSELoss

In [ ]:
criterion = nn.MSELoss()
mae_loss_criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


Обучение модели происходит в 50 эпох с записью функции потерь

In [ ]:
n_epochs = 10
loss_history = []


Пайплайн обучения

In [ ]:
for epoch in range(n_epochs):
    epoch_loss = 0.0
    for batch_X, batch_y in train_dataloader:
        optimizer.zero_grad()
        y_pred = model(batch_X)
        loss = criterion(y_pred, batch_y.squeeze())
        loss.backward()
        optimizer.step()
        with torch.no_grad():
          loss = mae_loss_criterion(y_pred, batch_y.squeeze())
          epoch_loss += loss.item()



    loss_history.append(epoch_loss / len(train_dataloader))
    print(f"Epoch {epoch+1}/{n_epochs}, MAE Loss: {epoch_loss / len(train_dataloader)}")


Epoch 1/10, MAE Loss: 0.5405912869965738
Epoch 2/10, MAE Loss: 0.6444031519703257
Epoch 3/10, MAE Loss: 1.072093888347535
Epoch 4/10, MAE Loss: 1.2611305699816153
Epoch 5/10, MAE Loss: 1.2721837422077005
Epoch 6/10, MAE Loss: 1.2818600743645785
Epoch 7/10, MAE Loss: 1.299392309749719
Epoch 8/10, MAE Loss: 1.3122331187558558
Epoch 9/10, MAE Loss: 1.3150099945105258
Epoch 10/10, MAE Loss: 1.5433364917240744
